# Merge Extended Features — CC1 + CC2 only

**New notebook — does not touch any existing pipeline file.** Adds container
metrics that exist in the raw data but were never merged in:
`container_cpu_cfs_throttled_seconds_total`, `container_cpu_cfs_throttled_periods_total`
(direct CPU-throttling signal), `container_threads` (plausible pod-failure
signal), and `container_spec_memory_limit_bytes` (used to derive a
proximity-to-limit ratio, not the raw limit itself, since the limit is close
to constant per container).

**Scoped to CC1 and CC2 only** — matches the project's current CC2-only drift
scope (SC1/SC2 dropped from drift evaluation), so there's no reason to
reprocess them here.

**One planned feature was dropped after checking the data**:
`new_container_id`-based restart detection. Verified directly: across all of
`complex_case1_merged.csv`, every `cmdb_id` maps to exactly **one**
`new_container_id` for the entire dataset — it never changes, even across
pod-failure events in this collection. Not usable as a restart signal here.

**Critical implementation detail**: each raw per-KPI file has the same
duplicate-row artifact found earlier (`clean_and_split.ipynb`). Every file gets
deduplicated *independently* before joining — joining two tables that both have
duplicate keys would multiply rows together (128 × 128 duplicates → thousands
of spurious rows for one timestamp), not just add columns.

In [1]:
import pandas as pd
import numpy as np
import os

BASE     = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'
RAW_DIR  = os.path.join(BASE, 'data', 'raw')
MERGED_DIR = os.path.join(BASE, 'data', 'merged')
OUT_DIR  = os.path.join(BASE, 'experiments', 'models_extended', 'data', 'merged_extended')
os.makedirs(OUT_DIR, exist_ok=True)

CASES = {
    'complex_case1': ('Complex Case-1', 'Case-1'),
    'complex_case2': ('Complex Case-2', 'Case-2'),
}

NEW_KPIS = {
    'kpi_container_cpu_cfs_throttled_seconds_total.csv':  'container_cpu_cfs_throttled_seconds_total',
    'kpi_container_cpu_cfs_throttled_periods_total.csv':  'container_cpu_cfs_throttled_periods_total',
    'kpi_container_threads.csv':                          'container_threads',
    'kpi_container_spec_memory_limit_bytes.csv':          'container_spec_memory_limit_bytes',
}

print('Paths configured. New KPIs to add:', list(NEW_KPIS.values()))

Paths configured. New KPIs to add: ['container_cpu_cfs_throttled_seconds_total', 'container_cpu_cfs_throttled_periods_total', 'container_threads', 'container_spec_memory_limit_bytes']


## Merge each case: base (existing 7 metrics) + 4 new KPIs, dedup before every join

In [2]:
def load_kpi(path, value_name):
    d = pd.read_csv(path, low_memory=False)
    d = d[['timestamp', 'cmdb_id', 'value']].rename(columns={'value': value_name})
    before = len(d)
    d = d.sort_values(['cmdb_id', 'timestamp']).drop_duplicates(subset=['cmdb_id', 'timestamp'], keep='first')
    return d, before, len(d)

for case, (folder, subfolder) in CASES.items():
    print(f'=== {case} ===')
    base = pd.read_csv(os.path.join(MERGED_DIR, f'{case}_merged.csv'), low_memory=False)
    before = len(base)
    base = base.sort_values(['cmdb_id', 'timestamp']).drop_duplicates(subset=['cmdb_id', 'timestamp'], keep='first')
    print(f'  base (existing 7 metrics): {before:,} -> {len(base):,} rows after dedup')

    merged = base
    kpi_dir = os.path.join(RAW_DIR, folder, subfolder, 'metric', 'container')
    for fname, colname in NEW_KPIS.items():
        kpi_df, n_before, n_after = load_kpi(os.path.join(kpi_dir, fname), colname)
        print(f'    {colname:45s}: {n_before:,} -> {n_after:,} rows after dedup')
        merged = merged.merge(kpi_df, on=['cmdb_id', 'timestamp'], how='left')

    n_missing = merged[list(NEW_KPIS.values())].isnull().sum()
    print(f'  merged shape: {merged.shape}')
    print(f'  missing values per new column:\n{n_missing}')

    out_path = os.path.join(OUT_DIR, f'{case}_merged_extended.csv')
    merged.to_csv(out_path, index=False)
    print(f'  saved -> {out_path}\n')

=== complex_case1 ===
  base (existing 7 metrics): 318,445 -> 223,830 rows after dedup
    container_cpu_cfs_throttled_seconds_total    : 224,330 -> 223,787 rows after dedup
    container_cpu_cfs_throttled_periods_total    : 224,330 -> 223,787 rows after dedup
    container_threads                            : 224,399 -> 223,830 rows after dedup
    container_spec_memory_limit_bytes            : 223,295 -> 222,836 rows after dedup
  merged shape: (223830, 14)
  missing values per new column:
container_cpu_cfs_throttled_seconds_total     43
container_cpu_cfs_throttled_periods_total     43
container_threads                              0
container_spec_memory_limit_bytes            994
dtype: int64
  saved -> c:\Users\jthar\Documents\Claude\Projects\module3\data\merged_extended\complex_case1_merged_extended.csv

=== complex_case2 ===
  base (existing 7 metrics): 94,932 -> 77,787 rows after dedup
    container_cpu_cfs_throttled_seconds_total    : 77,922 -> 77,787 rows after dedup
    cont

## Verify: row counts match the original merged files (join should only add columns, not rows or NaNs from fan-out)

In [3]:
for case in CASES:
    orig = pd.read_csv(os.path.join(MERGED_DIR, f'{case}_merged.csv'), low_memory=False)
    orig_dedup = orig.drop_duplicates(subset=['cmdb_id', 'timestamp'], keep='first')
    ext = pd.read_csv(os.path.join(OUT_DIR, f'{case}_merged_extended.csv'), low_memory=False)
    print(f'{case}: original dedup rows={len(orig_dedup):,}  extended rows={len(ext):,}  match={len(orig_dedup) == len(ext)}')
    print(f'  columns: {list(ext.columns)}')

complex_case1: original dedup rows=223,830  extended rows=223,830  match=True
  columns: ['timestamp', 'cmdb_id', 'new_container_id', 'container_cpu_usage_seconds_total', 'container_cpu_system_seconds_total', 'container_cpu_user_seconds_total', 'container_memory_usage_bytes', 'container_memory_working_set_bytes', 'container_memory_rss', 'container_memory_cache', 'container_cpu_cfs_throttled_seconds_total', 'container_cpu_cfs_throttled_periods_total', 'container_threads', 'container_spec_memory_limit_bytes']
complex_case2: original dedup rows=77,787  extended rows=77,787  match=True
  columns: ['timestamp', 'cmdb_id', 'new_container_id', 'container_cpu_usage_seconds_total', 'container_cpu_system_seconds_total', 'container_cpu_user_seconds_total', 'container_memory_usage_bytes', 'container_memory_working_set_bytes', 'container_memory_rss', 'container_memory_cache', 'container_cpu_cfs_throttled_seconds_total', 'container_cpu_cfs_throttled_periods_total', 'container_threads', 'container_sp